Environment Setup and Authentication

In [1]:
import sys
import os

# Upgrade Hugging Face ecosystem safely
!{sys.executable} -m pip install --upgrade --quiet transformers accelerate huggingface_hub datasets

import torch
import math
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login

# Hardware check
device_count = torch.cuda.device_count()
print(f"PyTorch {torch.__version__} | Detected {device_count} GPUs.")

# Authenticatation
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
    login(token=HF_TOKEN)
    print("Authenticated via Kaggle Secrets.")
except Exception as e:
    HF_TOKEN = "" 
    login(token=HF_TOKEN)
    print("Authenticated via explicit string.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 67.1 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.9/784.9 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 72.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 27.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
PyTorch 2.10.0+cu128 | Detected 2 GPUs.
Authenticated via explicit string.


Evaluation Metrices

In [2]:
class IntrinsicEvaluator:
    def __init__(self, model, tokenizer, device="cuda:0"):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        
    def get_lengths(self, text: str):
        """Calculates text-intrinsic properties for normalization."""
        if not isinstance(text, str):
            text = str(text)
        num_chars = max(1, len(text))
        num_bytes = max(1, len(text.encode('utf-8')))
        num_words = max(1, len(text.strip().split()))
        return num_chars, num_bytes, num_words

    def evaluate_sequence(self, text: str):
        """Calculates raw NLL and normalizes it into BPB, BPC, and BPW."""
        chars, bytes_count, words = self.get_lengths(text)
        inputs = self.tokenizer(text, return_tensors="pt").to(self.device)
        
        with torch.no_grad():
            outputs = self.model(**inputs, labels=inputs["input_ids"])
            # HF CausalLM loss is the average negative log-likelihood per token
            seq_len = inputs["input_ids"].shape[1]
            neg_log_likelihood = outputs.loss.item() * (seq_len - 1)
            
        # Calculate Tokenizer-Independent Metrics
        bpb = neg_log_likelihood / (bytes_count * math.log(2))
        bpc = neg_log_likelihood / (chars * math.log(2))
        bpw = neg_log_likelihood / (words * math.log(2))
        
        return {
            "nll": neg_log_likelihood,
            "bpb": bpb, 
            "bpc": bpc, 
            "bpw": bpw
        }
print("IntrinsicEvaluator class defined.")

IntrinsicEvaluator class defined.


Standardized Data Loading & Preprocessing

In [3]:
diverse_sentences_path = "/kaggle/input/datasets/shanilpraveen/intrinsic/diverse_sentences.csv"
sinhala_mixed_path = "/kaggle/input/datasets/shanilpraveen/intrinsic/sinhala_mixed_dataset.csv"

try:
    diverse_df = pd.read_csv(diverse_sentences_path).dropna()
    mixed_df = pd.read_csv(sinhala_mixed_path).dropna()
    print(f"Loaded Diverse Sentences: {len(diverse_df)} parallel pairs.")
    print(f"Loaded Mixed Script: {len(mixed_df)} sentences.")
except Exception as e:
    print(f"Data loading failed. Check your file paths. Error: {e}")

Loaded Diverse Sentences: 500 parallel pairs.
Loaded Mixed Script: 500 sentences.


Model & Tokenizer Initialization

In [4]:
model_id = "nvidia/Minitron-8B-Base"
print(f"Loading {model_id}...")

tokenizer = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model natively in fp16 across both GPUs
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    token=HF_TOKEN,
    low_cpu_mem_usage=True
)
model.eval()

# Initialize our custom evaluator
evaluator = IntrinsicEvaluator(model, tokenizer)

print(f"Model loaded.")
print(f"Precision: {model.dtype} | Device Map: {model.hf_device_map}")

Loading nvidia/Minitron-8B-Base...


config.json:   0%|          | 0.00/635 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/180k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 18.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 16.5GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/324 [00:00<?, ?it/s]

Model loaded.
Precision: torch.bfloat16 | Device Map: {'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 0, 'model.layers.8': 0, 'model.layers.9': 0, 'model.layers.10': 0, 'model.layers.11': 1, 'model.layers.12': 1, 'model.layers.13': 1, 'model.layers.14': 1, 'model.layers.15': 1, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.layers.28': 1, 'model.layers.29': 1, 'model.layers.30': 1, 'model.layers.31': 1, 'model.norm': 1, 'model.rotary_emb': 1, 'lm_head': 1}


Pilot Run

In [5]:
import csv

pilot_output_path = 'pilot_intrinsic_results.csv'
print("Starting Pilot Run (First 20 items)...")

# Open CSV in append mode to protect against runtime crashes
with open(pilot_output_path, mode='w', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    # Write standardized header
    writer.writerow([
        'item_index', 
        'unicode_nll', 'unicode_bpb', 'unicode_bpc', 'unicode_bpw',
        'romanized_nll', 'romanized_bpb', 'romanized_bpc', 'romanized_bpw'
    ])
    
    # Slice the first 20 rows of our loaded Kaggle dataframe
    pilot_sample = diverse_df.head(20)
    
    for index, row in pilot_sample.iterrows():
        # 1. Evaluate Native Unicode
        uni_metrics = evaluator.evaluate_sequence(row['sinhala_unicode'])
        
        # 2. Evaluate Romanized (Singlish)
        rom_metrics = evaluator.evaluate_sequence(row['sinhala_romanized'])
        
        # Log to disk
        writer.writerow([
            index, 
            uni_metrics['nll'], uni_metrics['bpb'], uni_metrics['bpc'], uni_metrics['bpw'],
            rom_metrics['nll'], rom_metrics['bpb'], rom_metrics['bpc'], rom_metrics['bpw']
        ])
        
        if (index + 1) % 5 == 0:
            print(f"Processed {index + 1}/20 parallel pairs...")

print(f"\n Pilot run complete. Results saved to {pilot_output_path}")

# Display verification preview
preview_df = pd.read_csv(pilot_output_path)
print("\nFirst 3 rows of calculated metrics:")
display(preview_df.head(3))

Starting Pilot Run (First 20 items)...
Processed 5/20 parallel pairs...
Processed 10/20 parallel pairs...
Processed 15/20 parallel pairs...
Processed 20/20 parallel pairs...

 Pilot run complete. Results saved to pilot_intrinsic_results.csv

First 3 rows of calculated metrics:


,item_index,unicode_nll,unicode_bpb,unicode_bpc,unicode_bpw,romanized_nll,romanized_bpb,romanized_bpc,romanized_bpw
0,0,88.676132,1.640162,4.264421,21.322103,90.679750,3.442716,3.442716,21.803871
1,1,65.808623,1.438512,3.955907,23.735444,70.256989,3.619979,3.619979,25.339852
2,2,85.090433,1.479031,3.959985,20.459924,78.668556,3.152632,3.152632,18.915789


Full Intrinsic Evaluation

In [6]:
import csv
import pandas as pd
from tqdm import tqdm

# Evaluate Parallel Diverse Sentences (Unicode vs Romanized)
full_diverse_output = 'minitron_8b_diverse_intrinsic.csv'
print(f"Starting full parallel evaluation on {len(diverse_df)} items...")

with open(full_diverse_output, mode='w', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    writer.writerow([
        'item_index',
        'unicode_nll', 'unicode_bpb', 'unicode_bpc', 'unicode_bpw',
        'romanized_nll', 'romanized_bpb', 'romanized_bpc', 'romanized_bpw'
    ])

    for index, row in tqdm(diverse_df.iterrows(), total=len(diverse_df), desc="Processing Parallel Pairs"):
        uni_metrics = evaluator.evaluate_sequence(row['sinhala_unicode'])
        rom_metrics = evaluator.evaluate_sequence(row['sinhala_romanized'])

        writer.writerow([
            index,
            uni_metrics['nll'], uni_metrics['bpb'], uni_metrics['bpc'], uni_metrics['bpw'],
            rom_metrics['nll'], rom_metrics['bpb'], rom_metrics['bpc'], rom_metrics['bpw']
        ])

print(f"Diverse evaluation complete. Saved to {full_diverse_output}")

# Evaluate Authentic Mixed-Script Sentences
full_mixed_output = 'minitron_8b_mixed_intrinsic.csv'
mixed_text_col = 'text' if 'text' in mixed_df.columns else mixed_df.columns[0]
print(f"\nStarting full mixed-script evaluation on {len(mixed_df)} items (column: '{mixed_text_col}')...")

with open(full_mixed_output, mode='w', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    writer.writerow(['item_index', 'mixed_nll', 'mixed_bpb', 'mixed_bpc', 'mixed_bpw'])

    for index, row in tqdm(mixed_df.iterrows(), total=len(mixed_df), desc="Processing Mixed Script"):
        mix_metrics = evaluator.evaluate_sequence(row[mixed_text_col])
        writer.writerow([
            index,
            mix_metrics['nll'], mix_metrics['bpb'], mix_metrics['bpc'], mix_metrics['bpw']
        ])

print(f"Mixed-script evaluation complete. Saved to {full_mixed_output}")

# Aggregate and Display Unicode vs Romanized Summary
def calculate_full_aggregates(csv_path, model_label="Minitron-8B-Base"):
    df = pd.read_csv(csv_path)

    summary = {
        'Metric': ['Bits-Per-Byte (BPB)', 'Bits-Per-Character (BPC)', 'Bits-Per-Word (BPW)'],
        'Unicode (Native)': [
            df['unicode_bpb'].mean(),
            df['unicode_bpc'].mean(),
            df['unicode_bpw'].mean()
        ],
        'Romanized (Singlish)': [
            df['romanized_bpb'].mean(),
            df['romanized_bpc'].mean(),
            df['romanized_bpw'].mean()
        ]
    }

    results_df = pd.DataFrame(summary)
    results_df['Degradation Factor'] = results_df['Romanized (Singlish)'] / results_df['Unicode (Native)']

    print("==========================================================")
    print(f"      {model_label} INTRINSIC EVALUATION SUMMARY")
    print("==========================================================")
    print(results_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print("==========================================================")

    styled = results_df.style.format({
        'Unicode (Native)': '{:.4f}',
        'Romanized (Singlish)': '{:.4f}',
        'Degradation Factor': '{:.2f}×'
    }).set_caption(f"{model_label} — Unicode vs Romanized Sinhala").background_gradient(
        subset=['Degradation Factor'], cmap='Reds'
    )
    display(styled)

    return results_df

full_summary = calculate_full_aggregates(full_diverse_output)

Starting full parallel evaluation on 500 items...


Processing Parallel Pairs: 100%|██████████| 500/500 [09:09<00:00,  1.10s/it]


Diverse evaluation complete. Saved to minitron_8b_diverse_intrinsic.csv

Starting full mixed-script evaluation on 500 items (column: 'text')...


Processing Mixed Script: 100%|██████████| 500/500 [06:26<00:00,  1.29it/s]

Mixed-script evaluation complete. Saved to minitron_8b_mixed_intrinsic.csv
      Minitron-8B-Base INTRINSIC EVALUATION SUMMARY
                  Metric  Unicode (Native)  Romanized (Singlish)  Degradation Factor
     Bits-Per-Byte (BPB)            1.6090                3.4652              2.1536
Bits-Per-Character (BPC)            4.2538                3.4730              0.8165
     Bits-Per-Word (BPW)           22.7703               21.4245              0.9409


,Metric,Unicode (Native),Romanized (Singlish),Degradation Factor
0,Bits-Per-Byte (BPB),1.6090,3.4652,2.15×
1,Bits-Per-Character (BPC),4.2538,3.4730,0.82×
2,Bits-Per-Word (BPW),22.7703,21.4245,0.94×


In [7]:
def calculate_mixed_aggregates(csv_path, model_label="Minitron-8B-Base"):
    df = pd.read_csv(csv_path)

    summary = {
        'Metric': ['Bits-Per-Byte (BPB)', 'Bits-Per-Character (BPC)', 'Bits-Per-Word (BPW)'],
        'Mixed-Script': [
            df['mixed_bpb'].mean(),
            df['mixed_bpc'].mean(),
            df['mixed_bpw'].mean()
        ],
        'Std Dev': [
            df['mixed_bpb'].std(),
            df['mixed_bpc'].std(),
            df['mixed_bpw'].std()
        ]
    }

    results_df = pd.DataFrame(summary)

    print("==========================================================")
    print(f"      {model_label} MIXED-SCRIPT INTRINSIC SUMMARY")
    print(f"      ({len(df)} sentences)")
    print("==========================================================")
    print(results_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print("==========================================================")

    styled = results_df.style.format({
        'Mixed-Script': '{:.4f}',
        'Std Dev': '{:.4f}'
    }).set_caption(f"{model_label} — Mixed-Script Sinhala")
    display(styled)

    return results_df

mixed_summary = calculate_mixed_aggregates(full_mixed_output)

      Minitron-8B-Base MIXED-SCRIPT INTRINSIC SUMMARY
      (500 sentences)
                  Metric  Mixed-Script  Std Dev
     Bits-Per-Byte (BPB)        2.0986   0.4545
Bits-Per-Character (BPC)        4.0242   0.4209
     Bits-Per-Word (BPW)       24.4890   3.8981


,Metric,Mixed-Script,Std Dev
0,Bits-Per-Byte (BPB),2.0986,0.4545
1,Bits-Per-Character (BPC),4.0242,0.4209
2,Bits-Per-Word (BPW),24.4890,3.8981
